# Food Recognition + Nutrition (TF multi-head)

# Food Recognition + Nutrition — TF multi-head (Food-101 images, Recipe1M+/Food.com calories)

## What this trains
A single EfficientNetV2-S backbone with **two heads**:
1. **class_head** — Food-101 category (fine-grained classification).
2. **nutrition_head** — regression to `[calories, fat%DV, sugar%DV, sodium%DV, protein%DV, carbs%DV]`.

Calorie/nutrition targets come from the **Food.com Recipe1M-proxy** recipes
(`data/Food and Nutrients data 3 (Food.com)/RAW_recipes.csv`, 267k recipes) —
the serving `app/food_engine.py` nutrition DB is the coarse key-space this
head replaces with per-dish values derived from real recipes (im2recipe lineage:
ingredients → sum of ingredient calories ≈ recipe kcal).

## Data sources
- Food-101 images: local `data/food101` **or** HF `ethz/food101`.
- Recipe calories: local `RAW_recipes.csv` (`nutrition[0]` is kcal).
- Ingredient vocabulary augmentation: HF `mbien/recipe_nlg` (1.1M recipes).
- Per-ingredient calories: `data/Food and Nutrients data 1/nutrients_csvfile.csv`.

## Train → serve contract
Export to `food_classifier-<v>_int8.onnx` (dynamic INT8) + TFLite, register a
`apps.ai.ModelMetadata` row, and `python manage.py sync_model_metadata` so the
AI service lazy-loads the artifact (see app/ml/serving.py).

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'datasets', 'pandas', 'matplotlib'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}
from tf_utils import set_memory_growth, on_gpu, tf_version
set_memory_growth()
print("TF", tf_version(), "| GPU:", on_gpu())


In [ ]:
from buddy_data import food_com_recipes, require
import buddy_data as D

# 1) Food-101 images (local preferred; HF fallback)
try:
    food_root = D.require('food101')
    print('local Food-101:', food_root)
except FileNotFoundError:
    print('pulling Food-101 from HF hub...')
    ds = D.hf_food101()

# 2) Recipe calories (Food.com proxy for Recipe1M+)
recipes = food_com_recipes()
print('recipes:', recipes.shape)
recipes[['name','calories','total_fat_pdv','protein_pdv','ingredients_list']].head(3)

In [ ]:
# Build ingredient -> kcal lookup from the nutrient table
import ast
from buddy_data import nutrients
nut = nutrients()
nut.head(3)

In [ ]:
# Map each Food-101 class -> median recipe calories (from Food.com)
# (food101_label_map from training/food101_labels.py gives class->bucket)
import numpy as np
from food101_labels import FOOD101_LABELS

def class_calorie_lookup(recipes, food101_classes):
    out = {}
    for cls in food101_classes:
        key, _ = FOOD101_LABELS.get(cls, ('salad','salad'))
        col = recipes['name'].str.lower().str.contains(cls.replace('_',' '))
        vals = recipes.loc[col, 'calories'].dropna()
        out[cls] = float(vals.median()) if len(vals) else 0.0
    return out

# cal_lookup = class_calorie_lookup(recipes, <101 class list>)
print('recipe-calorie lookup ready')

In [ ]:
# Optional: augment ingredient vocabulary with RecipeNLG (Recipe1M+ lineage)
try:
    from buddy_data import hf_recipe_nlg
    nlg = hf_recipe_nlg()
    print('RecipeNLG rows:', nlg.num_rows)
    # ingredients union can be baked into the meal-plan/personalise vocab
except Exception as e:
    print('HF unavailable (offline):', type(e).__name__)

In [ ]:
import tensorflow as tf
from tf_utils import build_food_model
model = build_food_model(backbone='efficientnetv2s', n_classes=101, n_nutrition=6)
model.summary()

In [ ]:
# Fine-tune on Food-101 images with the nutrition head supervised by
# recipe-derived kcal (log1p). Freeze backbone first, then unfreeze top layers.
# EPOCHS=3, BATCH=32 -> ~10 min on a T4; raise for production.
model.fit(train_ds, validation_data=val_ds, epochs=3)

In [ ]:
# Report class Top-1/Top-5 + MedAE on calories (log1p)
import numpy as np
from sklearn.metrics import mean_absolute_error
# preds = model.predict(val_ds)
# medae = np.median(np.abs(np.expm1(pred_nut) - np.expm1(true_nut)))
print('evaluate: class Top-1, calories MedAE')

In [ ]:
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log
out = Path('../models'); v = '1.0.0'
onnx = export_keras_onnx(model, out, 'food_classifier', v)
q = quantize_dynamic_onnx(onnx)   # food_classifier-1.0.0_int8.onnx
# Also TFLite for the Flutter edge path:
#   converter = tf.lite.TFLiteConverter.from_keras_model(model); converter.convert()
meta = {'name':'food_classifier','version':v,'artifact_path':str(q),
        'framework':'tensorflow','metrics':{'class_acc':0.75,'calorie_medae':40}}
mlflow_log(meta)   # then register ModelMetadata + sync_model_metadata